# Introduzione

Il presente progetto è stato realizzato per l'esame di Massive Data Mining, si incentra sul dataset TMDB 5000 Movie Dataset, una raccolta di informazioni su 5000 film tra cui titoli, generi, budget, incassi, cast, regia e sinossi.

L'obiettivo principale del progetto è la realizzazione di un recommendation system cinematografico in grado di restituire, a partire da una query dell'utente, che può essere il titolo di un film anche scritto in modo approssimato, oppure una semplice descrizione a testo libero, i 10 candidati più pertinenti presenti nel dataset. La sfida principale consiste nel gestire in modo robusto l'imprecisione dell'input: errori di battitura, titoli parziali e query descrittive rappresentano scenari reali e frequenti che un sistema di ricerca efficace deve saper affrontare.

A tal fine sono stati esplorati e messi a confronto due approcci distinti: il fuzzy matching, basato sulla distanza di Levenshtein tramite la libreria `TheFuzz`, e la ricerca basata su `cosine similarity`, realizzati entrambi tramite embedding vettoriali generati dal modello `all-mpnet-base-v2` della libreria `Sentence Transformers`.

# Importazione delle librerie

Le librerie utilizzate nel codice sono:
- `ast` per leggere stringhe che sembrano liste o dizionari e trasformarle in oggetti
- `numpy` per calcoli numerici
- `pandas` per lavorare con tabelle di dati
- La classe `SentenceTransformer` dalla libreria `sentence_transformers` per usare un modello di intelligenza artificiale in grado di trasformare una frase in un vettore numerico che ne rappresenta il significato
- La funzione `cosine_similarity` dalla libreria `scikit-learn` per misurare quanto due vettori numerici puntano nella stessa direzione, restituendo un valore tra -1 e 1
- Il modulo `fuzz` dalla libreria `thefuzz` per avere funzioni di confronto tra stringhe di testo anche quando non sono identiche (fuzzy matching)

In [ ]:
import ast
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from thefuzz import fuzz

# Definizione dei parametri di configurazzione
Per rendere il progetto il più possibile flessibile sono stati definiti alcuni parametri di configurazione modificabili in modo dinamico in modo da gestire al meglio il funzionamento del programma, in particolare abbiamo:
- `TOP` definisce il numero di film da mostrare alla fine
- `CAST_LIMIT` indica quanti membri del cast preservare dalla pulizia
- `MODEL_NAME` ci dice quale modello di Sentence Transformer prendere
- `TITLE_THRESHOLD` definisce il livello di accuratezza del titolo da considerare per la modalità ricerca tramite tiolo
- `WEIGHT_TITLE_HIGH` indica il peso dello score del titolo sulla query in caso di ricerca per titolo
- `WEIGHT_TITLE_LOW` indica il peso dello score del titolo sulla query in caso di ricerca per descrizione

Inoltre, vengono elencate le `queries` che verranno usate per testare gli algoritmi con i corrispondenti film associati che dovrebbero trovare.

### La scelta del modello
Il modello scelto è `all-mpnet-base-v2`, un modello di **sentence embeddings** della libreria Sentence Transformers, progettato per convertire frasi e brevi paragrafi in vettori numerici che catturano il significato semantico del testo.

Il modello si basa su **MPNet** di Microsoft, un sistema che legge e analizza il testo in modo più accurato rispetto ai modelli precedenti come BERT. È stato "allenato" su un miliardo di coppie di frasi imparando a capire quali frasi hanno un significato simile e quali no.
Ogni testo in input viene trasformato in un vettore a 768 dimensioni che rappresenta il suo significato. Il limite di input di default è 384 word pieces.
Tra i modelli della stessa famiglia, `all-mpnet-base-v2` è quello che produce i risultati migliori in termini di qualità. Rispetto a versioni più leggere come `all-MiniLM-L6-v2`, capisce il testo in modo più preciso, ma è anche più lento e richiede più risorse computazionali.

In [2]:
TOP = 10
CAST_LIMIT = 5
MODEL_NAME = "all-mpnet-base-v2"

TITLE_THRESHOLD = 0.65
WEIGHT_TITLE_HIGH = 0.70
WEIGHT_TITLE_LOW = 0.25

queries = [
    # Titoli Completi
    ("Blade Runner", "Blade Runner"),
    ("The Dark Knight Rises", "The Dark Knight Rises"),
    ("Raging Bull", "Raging Bull"),
    ("Robocop", "Robocop"),
    # Titoli parziali o scritti male
    ("goffather", "The Godfather"),
    ("gost", "Ghost"),
    ("lord of the rings", "The Lord of the Rings: The Fellowship of the Ring"),
    ("montecristo", "The count of monte cristo"),
    ("the mtrax", "The Matrix"),
    ("big short", "The Big Short"),
    # Descrizioni
    ("science fiction movie with light sabers", "Star Wars"),
    ("animated movie about a rat who cooks", "Ratatouille"),
    ("superhero with iron suit", "Iron Man"),
    ("killer shark on a beach town", "Jaws"),
    ("space crew finds alien eggs on a planet", "Alien"),
    ("mel brooks parody of star wars with spaceships", "Spaceballs"),
    ("time travel DeLorean 1985", "Back to the Future"),
    ("tributes are chosen to compete in killing games", "The Hunger Games"),
    ("horror movie with a masked killer", "Halloween"),
    ("cyborg travels into the past to kill", "The Terminator"),
    # Attori
    ("emma stone, ryan gosling and steve carell", "Crazy, Stupid, Love."),
    ("matt damon and robin williams", "Good Will Hunting"),
    ("ryan gosling as a stuntman", "Drive"),
    ("margot robbie in a martin scorsese movie", "The Wolf of Wall Street"),
    ("science fiction movie by john carpenter", "The Thing"),
    ]

# Caricamento e merge dei dataset
In primis vengono caricati entrambi i file **CSV** tramite **Pandas** e vengono uniti tramite l'id univoco di ogni film. Questa operazione viene eseguita per evitare omonimie di film presenti nel database (es. Batman 1989 e Batman 1966).

In [3]:
df1 = pd.read_csv("data/tmdb_5000_movies.csv")
df2 = pd.read_csv("data/tmdb_5000_credits.csv")
df = pd.merge(df1, df2, left_on="id", right_on="movie_id")

# Helper functions
Successivamente vengono definite tutte le funzioni di sostegno alla ricerca e alla pulizia del dataset, ognuna con uno scopo diverso.
#### extract_names
La funzione `extract_names` estrae i nomi da una colonna JSON-like e li riporta come semplice testo, con la possibilità di applicare un limite al numero di nomi che vogliamo estrarre.
#### extract_list
La funzione `extract_list`, come la precedente, estrae i nomi da una colonna JSON-like, ma riporta sotto forma di lista, anche qui con la possibilità di applicare un limite al numero di nomi che vogliamo estrarre.
#### get_director
La funzione `get_director` estrae il nome del regista dalla colonna JSON-like, in modo da non avere inutilmente tutti i membri della crew che ha lavorato al film visto che non sono rilevanti ai fini della ricerca.
#### fuzzy_title_score
La funzione `fuzzy_title_score` sfrutta la libreria **TheFuzz** per lo string matching con la **Distanza di Levenshtein**. In particolare la funzione è stata creata per assegnare ad ogni titolo uno score da 0 a 1 in base alla somiglianza con la query che stiamo analizzando.
Questa funzione usa i parametri:
- `ratio` per un confronto carattere per carattere, stringa intera
- `partial_ratio` con cui cerca la query come sottostringa del titolo (ideale per titoli parziali)
- `token_set_ratio` grazie al quale si ignorano l'ordine delle parole ed i duplicati
- `token_sort_ratio` per ordinare le parole alfabeticamente prima di confrontarle

Prende il massimo tra questi valori che normalmente vanno da 0 a 100 e lo divide per 100 per rimanere nell'intervallo [0, 1]. Prendere il massimo tra questi valori garantisce che almeno uno dei criteri di matching "colpisca" nel caso migliore.
Una limitazione del calcolo della fuzzy search risiede nel fatto che titoli estremamente corti sono facilmente rintracciabili con qualsiasi ricerca basata sul titolo, per mitigare questo problema viene eseguito un controllo sulla lunghezza della query e del titolo per penalizzare titoli troppo corti su query più lunghe.

#### fuzzy_title_score_strict
La funzione `fuzzy_title_score_strict` è una versione con metriche più "strette" di `fuzzy_title_score`, usata solo per decidere se la query è un titolo o una descrizione, ma non per il ranking finale.
In questo caso viene escluso il `partial_ratio` dato che una query descrittiva lunga come *"thriller movie with a girl protagonist"* contiene parole comuni. Un titolo corto come *"The Girl"* o *"With"* viene trovato come sottostringa e ottiene *partial_ratio=100*, facendo scattare erroneamente la modalità titolo. `ratio` e `token_sort_ratio` richiedono invece una somiglianza globale tra le due stringhe, e non si innescano su query lunghe.

#### normalize
La funzione `normalize` esegue una *min-max normalization*, portando tutti i valori in [0, 1]. In primis calcola il valore minimo e massimo della serie per poi spostare tutti i valori facendo diventare il minimo 0 ed il massimo 1. Inoltre se il valore minimo corrisponde al valore massimo restituisce la serie senza alterarla, per evitare di eseguire l'operazione 0/0.

In [4]:
def extract_names(text, limit = None):
    try:
        items = ast.literal_eval(text)
        names = [item["name"] for item in items]
        if limit:
            names = names[:limit]
        return " ".join(names)
    except:
        return ""
    
def extract_list(text, limit = None):
    try:
        items = ast.literal_eval(text)
        names = [item["name"] for item in items]
        return names[:limit] if limit else names
    except:
        return []

def get_director(text):
    try:
        items = ast.literal_eval(text)
        for item in items:
            if item["job"] == "Director":
                return item["name"]
        return ""
    except:
        return ""

def fuzzy_title_score(query, title):
    q, t = query.lower(), title.lower()
    score = max(
        fuzz.ratio(q, t),
        fuzz.partial_ratio(q, t),
        fuzz.token_set_ratio(q, t),
        fuzz.token_sort_ratio(q, t),
    ) / 100.0

    if len(t) <= 3 and len(q) > len(t) * 3:
        len_ratio = min(len(t), len(q)) / max(len(t), len(q))
        score *= len_ratio

    return score

def fuzzy_title_score_strict(query, title):
    q, t = query.lower(), title.lower()
    return max(
        fuzz.ratio(q, t),
        fuzz.token_sort_ratio(q, t),
    ) / 100.0

def normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

# Prerocessing dataset
Per il preprocessing del dataset vengono eseguite diverse operazioni in modo da pulirlo e renderlo più comodo per eseguire la vera e propria ricerca.
In primo luogo viene estratto l'anno di pubblicazione del film dal campo `release_date`, in cui abbiamo una data in formato *YYYY-MM-DD*, ed inserito nella colonna `year`. In questo modo se abbiamo film omonimi possiamo distinguerli in base all'anno di rilascio, piuttosto che scartarli.
Successivamente viene ricavato il nome del regista semplicemente applicando alla colonna `crew` la funzione `get_director` definita precedentemente e riportando il risultato all'interno della nuova colonna `director`. Allo stesso modo estraiamo i nomi del cast con la funzione `extract_list` e li uniamo al nome del regista, dato che non ci è utile averli separati.

Eliminiamo le colonne per cui non è stato rilasciato il film controllando la colonna `status` per la stringa "Released", poiché potrebbero contenere dati mancanti o inesatti.

Vengono poi eliminate tutte quelle colonne che non sono utili alla nostra ricerca, ovvero: `title_y` (duplicata in seguito al merge), `tagline`, `homepage`, `runtime`, `budget`, `revenue`, `vote_count`, `vote_average`, `popularity`, `movie_id` (usiamo la colonna id al suo posto), `production_countries`, `spoken_languages`, `production_companies`, `director` (non più utile visto che abbiamo il suo nome insieme al cast) e `status`.
Dato che nella colonna `overview` è contenuta la sinossi del film è un dato preziosissimo per la ricerca tramite query, scartiamo le due righe che non presentano questa descrizione.
Rinominiamo poi la colonna `title_x` per farla tornare al nome originale `title`.

Come abbiamo fatto per le altre colonne estraiamo anche i contenuti delle colonne `genres` e `keywords` per eliminare il formato JSON-like e poterli usare nella nostra ricerca.
Viene poi sostituito il contenuto delle righe vuote nelle colonne che contengono stringhe con stringhe vuote per evitare errori durante la ricerca o l'utilizzo di qualche funzione.

Uniamo poi il contenuto di tutte le colonne che contengono testi e tag che possono aiutarci nella ricerca descrittiva di un film e andiamo a salvare l'unione di questi testi nella nuova colonna `semantic_text`. Tutto questo facendo particolare attenzione a preservare i contenuti così come sono e senza eseguire stemming dato che i sentence transformers sono addestrati su testo leggibile e deteriorano i propri risultati con forme ridotte.

Infine reimpostiamo gli indici del dataframe per evitare il disallineamento degli indici.

In [5]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year.astype("Int64")

df["director"] = df["crew"].apply(get_director)
df["cast"] = df["cast"].apply(lambda x: extract_list(x, limit=CAST_LIMIT))
df["cast"] = df["cast"] + df["director"].apply(lambda x: [x])

df = df[df["status"] == "Released"]

df = df.drop(["title_y", "tagline", "homepage", "runtime", "budget", "revenue", "vote_count", "vote_average", "popularity", "movie_id", "production_countries", "spoken_languages", "production_companies", "director", "status"], axis = 1)
df = df.dropna(subset=["overview"])
df = df.rename(columns={"title_x": "title"})

df["genres"] = df["genres"].apply(extract_list)
df["keywords"] = df["keywords"].apply(extract_names)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("")

df["semantic_text"] = (
    df["overview"] + " " +
    df["genres"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x)) + " " +
    df["keywords"] + " " +
    df["cast"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
)

df = df.reset_index(drop=True)

# Encoding del database
Arrivati a questo punto utilizziamo il modello `all-mpnet-base-v2` scelto precedentemente per eseguire l'encoding del nostro `semantic_text` con i sentence transformers. Tramite il parametro `batch_size=64` il modello elabora 64 testi alla volta in modo da essere più veloce.

La stessa operazione viene eseguita anche sui titoli in modo da poterli classificare usando la funzione con la **Cosine Similarity** piuttosto che sul **Fuzzy Search**. 

In [6]:
sentence_model = SentenceTransformer(MODEL_NAME)

print("Encoding del database in corso...")
db_vectors = sentence_model.encode(
    df["semantic_text"].tolist(),
    show_progress_bar=True,
    batch_size=64,
)
print("Encoding completato.\n")

print("Encoding dei titoli in corso...")
title_vectors = sentence_model.encode(
    df["title"].tolist(),
    show_progress_bar=True,
    batch_size=64,
)
print("Encoding completato.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9550.72it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding del database in corso...


Batches: 100%|██████████| 75/75 [00:41<00:00,  1.80it/s]


Encoding completato.

Encoding dei titoli in corso...


Batches: 100%|██████████| 75/75 [00:03<00:00, 23.30it/s]

Encoding completato.


# Funzione di ricerca
Ora entriamo nel pieno del progetto andando a definire la funzione di ricerca `search` che utilizzeremo per il vero e proprio confronto. Questa funzione, data una `query` (titolo anche approssimato, o stringa descrittiva), restituisce i `TOP` film più rilevanti ordinati per score totale. Qui è possibile anche selezionare la `search_mode` preferita tra la modalità `fuzzy` e la modalità `cosine` che sfruttano diversi algoritmi per la similarità dei contenuti del testo.
Lo scoring viene effettuato tramite dei pesi adattivi e l'aggiunta di un punteggio ulteriore in caso di presenza del nome di un attore o di un genere preciso.
A questo punto viene eseguito l'encoding anche della query che abbiamo ricevuto in input per poterla confrontare successivamente in base alla modalità scelta.

Qui entriamo nel vivo delle due modalità, se ci troviamo nella modalità `fuzzy` le operazioni saranno le seguenti:

Viene calcolata la soglia di somiglianza ai titoli tramite `fuzzy_title_score_strict`. Il valore deve essere maggiore o uguale al valore di `TITLE_THRESHOLD` per poter entrare nella modalità di ricerca tramite titolo, in cui il peso dello score di quest'ultimo è definito nella variabile `WEIGHT_TITLE_HIGH` (0,70), altrimenti sarà `WEIGHT_TITLE_LOW` (0.25). In questo modo il peso dello score della somiglianza semantica sarà complementare a quello dei titoli.
A questo punto viene eseguito il vero e proprio ranking dei titoli tramite `fuzzy_title_score` per consentire la rilevanza anche dei titoli parziali.

Se invece ci troviamo in modalità `cosine` le operazioni saranno le seguenti:

Viene calcolata la cosine similarity tra i vettori dei titoli ed il vettore della query, per poi trasformare la lista degli score appena ottenuti in una Series. Successivamente cerca e memorizza nella variabile `max_strict` il valore più alto presente in tutta la lista dei punteggi, per capire qual è il grado massimo di pertinenza trovato per quella specifica ricerca.
Infine, assegna gli score ai titoli e usa i punteggi ottenuti per definire il peso e la modalità di ricerca da applicare.

Se ci troviamo in modalità descrittiva vengono cercati inoltre nomi completi di attori e generi all'interno della query, in modo da poter aggiungere 0,25 (fino ad 1 per ogni categoria) allo score totale, questo perché si è ritenuto più importante il genere ed il nome di un attore in una ricerca piuttosto che una ricerca sommaria.

L'encoding della query eseguito precedentemente potrà finalmente essere confrontato tramite la **Cosine Similarity** con i vettori ottenuti dall'encoding del testo semantico.
Possiamo a questo punto normalizzare gli score ottenuti sia dalla similarità del titolo sia dalla similarità della semantica, per poi andare a sommare tutti i fattori con i loro rispettivi pesi.

Salviamo i nostri risultati in un nuovo dataframe chiamato `results`, che, oltre a possedere gli attributi del database originale `title`, `year` e `genres`, ottiene i nuovi attributi relativi ai diversi score calcolati.

Stampa, infine, la modalità di ricerca che è stata utilizzata con i pesi relativi che sono stati usati ed il valore di corrispondenza massima della query in input con i titoli del database. Tutto questo per consentirci di eseguire una piccola fase di debug nel caso in cui stiamo filtrando il database.

In [7]:
def search(query, search_mode="fuzzy", top = TOP):
    found_genres = pd.Series(np.zeros(len(df)), index=df.index)
    found_actors = pd.Series(np.zeros(len(df)), index=df.index)    

    q_vec = sentence_model.encode([query])
    
    if search_mode == "fuzzy":
        strict_scores = df["title"].apply(lambda t: fuzzy_title_score_strict(query, t))
        max_strict = strict_scores.max()
        title_scores = df["title"].apply(lambda t: fuzzy_title_score(query, t))
    else:
        title_cos = cosine_similarity(title_vectors, q_vec).flatten()
        strict_scores = pd.Series(title_cos, index=df.index)
        max_strict = strict_scores.max()
        title_scores = strict_scores

    if max_strict >= TITLE_THRESHOLD:
        w_title = WEIGHT_TITLE_HIGH
        mode = "titolo"
    else:
        w_title = WEIGHT_TITLE_LOW
        mode = "descrittiva"
        df["genre_found"] = df["genres"].apply(lambda genres: [g for g in genres if g.lower() in query.lower()])
        df["genre_score_found"] = df["genre_found"].apply(lambda x: len([i for i in x if i]) * 0.25)
        found_genres = df["genre_score_found"].clip(upper=1.0)

        df["cast_found"] = df["cast"].apply(lambda names: [n for n in names if n.lower() in query.lower()])
        df["cast_score_found"] = df["cast_found"].apply(lambda x: len([i for i in x if i]) * 0.25)
        found_actors = df["cast_score_found"].clip(upper=1.0)

    w_semantic = 1.0 - w_title

    sem_scores = cosine_similarity(db_vectors, q_vec).flatten()

    sem_norm = normalize(pd.Series(sem_scores, index=df.index))
    title_norm = normalize(pd.Series(title_scores, index=df.index))
    total = w_semantic * sem_norm + w_title * title_norm + found_actors + found_genres

    results = df[["title", "year", "genres"]].copy()
    results["score_semantico"] = sem_norm.values
    results["score_titolo"] = title_norm.values
    results["cast_score_found"] = found_actors.values
    results["genre_score_found"] = found_genres.values
    results["score_totale"] = total.values

    if top != len(df):
        print(
            f"[search] modalità: {mode} | "
            f"w_titolo={w_title:.2f} | w_semantico={w_semantic:.2f} | "
            f"max_strict={max_strict:.2f}"
        )

    return results.sort_values("score_totale", ascending=False).head(top)

# Esecuzione
A questo punto non resta che l'esecuzione del codice andando a richiamare la funzione `search`, prima in modalità `fuzzy` e poi per la modalità `cosine`, per le diverse query che diamo in input e stampare i risultati per verificare il corretto funzionamento del codice.

In [8]:
for query in queries:
    query = query[0]
    top = search(query, search_mode="fuzzy")
    
    print(f'\nTop {TOP} film per: "{query}"\n')
    print(f'{"#":<3} {"Titolo":<45} {"Anno":<6} {"Totale":>7}  {"Semantico":>9}  {"Titolo":>7} {"Genere":>7} {"Cast":>7} {"Generi":<10}')
    print("-" * 135)
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        print(
            f'{rank:<3} {str(row["title"])[:44]:<45} {str(row["year"]):<6} '
            f'{row["score_totale"]:>7.3f}  {row["score_semantico"]:>9.3f}  {row["score_titolo"]:>7.3f} {row["genre_score_found"]:>7.3f} {row["cast_score_found"]:>7.3f} {str(", ".join(row["genres"][:3])):<10}'
        )

[search] modalità: titolo | w_titolo=0.70 | w_semantico=0.30 | max_strict=1.00

Top 10 film per: "Blade Runner"

#   Titolo                                        Anno    Totale  Semantico   Titolo  Genere    Cast Generi    
---------------------------------------------------------------------------------------------------------------------------------------
1   Blade Runner                                  1982     1.000      1.000    1.000   0.000   0.000 Science Fiction, Drama, Thriller
2   Blade                                         1998     0.902      0.673    1.000   0.000   0.000 Horror, Action
3   Runner Runner                                 2013     0.895      0.650    1.000   0.000   0.000 Crime, Thriller, Drama
4   Blade II                                      2002     0.806      0.679    0.860   0.000   0.000 Fantasy, Horror, Action
5   The Maze Runner                               2014     0.749      0.582    0.820   0.000   0.000 Action, Mystery, Science Fiction
6   Th

In [9]:
for query in queries:
    query = query[0]
    top = search(query, search_mode="cosine")
    
    print(f'\nTop {TOP} film per: "{query}"\n')
    print(f'{"#":<3} {"Titolo":<45} {"Anno":<6} {"Totale":>7}  {"Semantico":>9}  {"Titolo":>7} {"Genere":>7} {"Cast":>7} {"Generi":<10}')
    print("-" * 135)
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        print(
            f'{rank:<3} {str(row["title"])[:44]:<45} {str(row["year"]):<6} '
            f'{row["score_totale"]:>7.3f}  {row["score_semantico"]:>9.3f}  {row["score_titolo"]:>7.3f} {row["genre_score_found"]:>7.3f} {row["cast_score_found"]:>7.3f} {str(", ".join(row["genres"][:3])):<10}'
        )

[search] modalità: titolo | w_titolo=0.70 | w_semantico=0.30 | max_strict=1.00

Top 10 film per: "Blade Runner"

#   Titolo                                        Anno    Totale  Semantico   Titolo  Genere    Cast Generi    
---------------------------------------------------------------------------------------------------------------------------------------
1   Blade Runner                                  1982     1.000      1.000    1.000   0.000   0.000 Science Fiction, Drama, Thriller
2   Terminator Salvation                          2009     0.703      0.726    0.693   0.000   0.000 Action, Science Fiction, Thriller
3   Sicario                                       2015     0.695      0.748    0.673   0.000   0.000 Action, Crime, Drama
4   Blade II                                      2002     0.692      0.679    0.698   0.000   0.000 Fantasy, Horror, Action
5   Interstellar                                  2014     0.676      0.595    0.710   0.000   0.000 Adventure, Drama, Scie

# Validazione
Per validare l'approccio migliore eseguiamo nuovamente per tutte le query il nostro codice, andando a verificare il loro funzionamento tramite la posizione del film che ci aspettiamo di ottenere.
Questo viene fatto andando a contare le varie volte in cui l'indice di un algoritmo è posizionato più in alto del suo competitor.

Come è possibile notare nella maggior parte dei casi si verifica un pareggio fra i due approcci, tuttavia, in 6 casi su 25 è l'algoritmo di fuzzy search a vincere il duello. Questa cosa è particolarmente visibile nella ricerca dei titoli parziali o negli errori di battitura, in cui l'approccio dato dalla cosine arriva in posizioni estremamente basse rispetto alla ricerca fuzzy.

È dunque a mio parere l'algoritmo `fuzzy` ad avere la meglio sul rivale, e risulta più valido rispetto all'approccio tramite `cosine`.

In [10]:
fuzzy_counter = 0
cosine_counter = 0
same_counter = 0

print(f'{"Query":<45} {"Titolo ricercato":<45} {"Fuzzy":>7} {"Cosine":>7}')
print("-" * 108)
for query in queries:

    try:
        fuzzy_results = search(query[0], search_mode="fuzzy", top=len(df)).reset_index(drop=True)
        fuzzy_position = fuzzy_results[fuzzy_results["title"].str.lower() == query[1].lower()].index[0] + 1

        cosine_results = search(query[0], search_mode="cosine", top=len(df)).reset_index(drop=True)
        cosine_position = cosine_results[cosine_results["title"].str.lower() == query[1].lower()].index[0] + 1
    except IndexError:
        print(f"Film {query[1]} non trovato.")

    if fuzzy_position < cosine_position:
        fuzzy_counter += 1
    elif fuzzy_position > cosine_position:
        cosine_counter += 1
    else:
        same_counter += 1

    print(f"{query[0][:44]:<45} {query[1][:44]:<45} {fuzzy_position:>7} {cosine_position:>7}")
    
print("\nRisultati:\n"
      f"Fuzzy migliore in {fuzzy_counter}/25 casi\n"
      f"Cosine migliore in {cosine_counter}/25 casi\n"
      f"Pari in {same_counter}/25 casi"
      )

Query                                         Titolo ricercato                                Fuzzy  Cosine
------------------------------------------------------------------------------------------------------------
Blade Runner                                  Blade Runner                                        1       1
The Dark Knight Rises                         The Dark Knight Rises                               1       1
Raging Bull                                   Raging Bull                                         1       1
Robocop                                       Robocop                                             1       1
goffather                                     The Godfather                                       4    4046
gost                                          Ghost                                               6    2513
lord of the rings                             The Lord of the Rings: The Fellowship of the        1       1
montecristo                

## Metriche di Valutazione
Per valutare i modelli, oltre la definizione della miglior posizione in classifica, possiamo utilizzare due metriche importanti: Mean Reciprocal Rank (MRR) e Hit@K. Per calcolare queste metriche definiamo la funzione evaluate.

Il Mean Reciprocal Rank è un indice statistico usato per valutare un processo che produce una lista di possibili risposte ad una interrogazione, ordinate per probabilità di correttezza. Si calcola semplicemente facendo:
$$MRR=\frac{\sum_{i=1}^Q{\frac{1}{rank_i}}}{Q}$$
Dove scorriamo da 1 a Q le query del nostro insieme.

L'Hit@K, invece, controlla se il film atteso è presente nei top-K risultati, in questo caso controlliamo per K = [1, 5, 10].

In [11]:
def evaluate(queries, mode):
    reciprocal_ranks = []
    k_values=[1, 5, 10]
    hits = {k: [] for k in k_values}

    for query, expected_title in queries:
        results = search(query, search_mode=mode, top=len(df))
        titles = results["title"].str.lower().tolist()

        rank = titles.index(expected_title.lower()) + 1

        reciprocal_ranks.append(1 / rank if rank else 0)
        for k in k_values:
            hits[k].append(1 if rank and rank <= k else 0)

    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    hit_k = {k: sum(hits[k]) / len(hits[k]) for k in k_values}
    return {"MRR": round(mrr, 4), **{f"Hit@{k}": round(v, 4) for k, v in hit_k.items()}}

Come possiamo vedere da questi score il punteggio della fuzzy resta ancora in vantaggio, i punteggi in cui il delta tra le due modalità è basso è nell'Hit@1 e nell'MRR. Negli altri risultati abbiamo un distacco più significativo.

In [12]:
fuzzy_scores  = evaluate(queries, mode="fuzzy")
cosine_scores = evaluate(queries, mode="cosine")

print(f'{"Modalità":<10} {"MRR":>8} {"Hit@1":>8} {"Hit@5":>8} {"Hit@10":>8}')
print("-" * 47)
print(f'{"Fuzzy":<10} {fuzzy_scores["MRR"]:>8} {fuzzy_scores["Hit@1"]:>8} {fuzzy_scores["Hit@5"]:>8} {fuzzy_scores["Hit@10"]:>8}')
print(f'{"Cosine":<10} {cosine_scores["MRR"]:>8} {cosine_scores["Hit@1"]:>8} {cosine_scores["Hit@5"]:>8} {cosine_scores["Hit@10"]:>8}')

Modalità        MRR    Hit@1    Hit@5   Hit@10
-----------------------------------------------
Fuzzy        0.8247     0.76     0.96      1.0
Cosine       0.7536     0.72      0.8     0.84
